In [ ]:
import os
import hashlib
import pandas as pd

# =========================
# 0) PATH CONFIG
# =========================
STAGE2_PATH = "../data/raw/harmful/stage2_data.csv"   # 첨부 파일 경로로 수정
OUT_DIR = "../data/impact"          # 출력 폴더
TARGET_N = 500

# =========================
# 1) PARAMETERS (재현 가능한 기준)
# =========================
MIN_LEN = 10
HARM_SCORE_MIN = 4   # "유해표현 과다/복합" 임계값

CATEGORIES = ["정치", "경제", "사회", "생활&문화", "IT&과학", "세계", "연예"]

# =========================
# 2) LOAD
# =========================
df = pd.read_csv(STAGE2_PATH)

required_cols = ["문장", "정상"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"필수 컬럼 누락: {missing}")

harm_cols = [c for c in df.columns if c not in ["문장", "정상"]]
if not harm_cols:
    raise ValueError("유해 라벨 컬럼을 찾지 못했습니다. (정상 제외 나머지 컬럼)")

# =========================
# 3) FEATURE ENGINEERING
# =========================
df["문장"] = df["문장"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df["문장"].str.len() >= MIN_LEN].copy()

df["harm_score"] = df[harm_cols].sum(axis=1)

# 임팩트4 후보: 정상=0 & 복합 유해(harm_score>=임계값)
cand = df[(df["정상"] == 0) & (df["harm_score"] >= HARM_SCORE_MIN)].copy()

# 중복 제거
cand = cand.drop_duplicates(subset=["문장"]).copy()

if len(cand) < TARGET_N:
    raise RuntimeError(
        f"후보 샘플이 부족합니다. cand={len(cand)} < TARGET_N={TARGET_N}. "
        f"HARM_SCORE_MIN을 낮추거나(MIN_LEN 포함) 기준을 완화하세요."
    )

# dominant label(최대값 컬럼명)로 다양성 확보
cand["dominant"] = cand[harm_cols].idxmax(axis=1)

def assign_category(text: str) -> str:
    h = int(hashlib.md5(text.encode("utf-8")).hexdigest(), 16)
    return CATEGORIES[h % len(CATEGORIES)]

cand["newsCategory"] = cand["문장"].apply(assign_category)
cand["newsSubcategory"] = cand["newsCategory"]

# 제목은 "유해 유형"을 명시해 모델이 신호를 학습하기 쉽게 구성(본문은 원문 유지)
cand["newsTitle"] = cand["dominant"].apply(lambda x: f"[주의] {x} 표현 논란")
cand["newsContent"] = cand["문장"]
cand["useType"] = 0  # 원천에 없는 필드이므로 '파생'임을 산출물에 명시 권장

# =========================
# 4) STRATIFIED SAMPLING (dominant 라벨 균형 + 카테고리 라운드로빈)
# =========================
dom_counts = cand["dominant"].value_counts()
labels = list(dom_counts.index)
base = TARGET_N // len(labels)

# 1차 할당: 라벨별 균등, 단 희소 라벨은 보유량 한도
alloc = {lab: min(base, int(dom_counts[lab])) for lab in labels}
remaining = TARGET_N - sum(alloc.values())

# 잔여분은 여유(spare)가 큰 라벨부터 1개씩 추가
spare = {lab: int(dom_counts[lab]) - alloc[lab] for lab in labels}
while remaining > 0 and any(v > 0 for v in spare.values()):
    lab = max(spare, key=lambda k: spare[k])
    if spare[lab] <= 0:
        break
    alloc[lab] += 1
    spare[lab] -= 1
    remaining -= 1

parts = []
for lab, n in alloc.items():
    sub = cand[cand["dominant"] == lab].copy()

    # 카테고리별 라운드로빈으로 뽑아 분포를 최대한 고르게
    groups = {c: sub[sub["newsCategory"] == c].sample(frac=1, random_state=42) for c in CATEGORIES}
    picked_rows = []
    i = 0
    while len(picked_rows) < n:
        c = CATEGORIES[i % len(CATEGORIES)]
        if len(groups[c]) > 0:
            row = groups[c].iloc[0]
            picked_rows.append(row)
            groups[c] = groups[c].iloc[1:]
        i += 1
        if all(len(g) == 0 for g in groups.values()):
            break

    parts.append(pd.DataFrame(picked_rows))

sel = pd.concat(parts, ignore_index=True)

# 안전장치: 정확히 TARGET_N 확보(부족하면 단순 랜덤 보충)
if len(sel) < TARGET_N:
    need = TARGET_N - len(sel)
    rest = cand[~cand["문장"].isin(sel["문장"])]  # 이미 뽑힌 문장 제외 (ignore_index 후에는 index로 구분 불가)
    rest = rest.sample(n=need, random_state=42)
    sel = pd.concat([sel, rest], ignore_index=True)

# 최종 중복 제거 및 TARGET_N 맞춤
sel = sel.drop_duplicates(subset=["newsContent"]).head(TARGET_N).copy()

# =========================
# 5) EXPORT (요구 컬럼 5개만)
# =========================
out = sel[["newsCategory", "newsSubcategory", "newsTitle", "newsContent", "useType"]].copy()

os.makedirs(OUT_DIR, exist_ok=True)
csv_path = os.path.join(OUT_DIR, "impact4_dataset_500.csv")
json_path = os.path.join(OUT_DIR, "impact4_dataset_500.json")

out.to_csv(csv_path, index=False, encoding="utf-8-sig")
out.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"[OK] rows={len(out)}")
print(f"- CSV : {csv_path}")
print(f"- JSON: {json_path}")
print("\n[CHECK] category distribution:\n", out["newsCategory"].value_counts())
print("\n[CHECK] title example:\n", out["newsTitle"].head(5).tolist())


In [ ]:
import os
import re
import json
import hashlib
import random
import pandas as pd

STAGE2_PATH = "../data/raw/harmful/stage2_data.csv"   # 첨부 파일 경로로 수정
OUT_DIR = "../data/impact"
TARGET_N = 500

MIN_LEN = 10

# 운영 관점 임팩트4를 더 강하게 만들고 싶으면 아래처럼 "고위험 포함" 조건을 쓰셔도 됩니다.
# (컬럼명이 정확히 있는지 확인 후 사용)
HIGH_RISK_LABELS_CANDIDATES = ["폭력위협/범죄조장", "외설", "성혐오", "인종/지역", "종교"]

CATEGORIES = ["정치", "경제", "사회", "생활&문화", "IT&과학", "세계", "연예"]

# 제목 템플릿 풀(최소 40개 이상 권장)
TITLE_TEMPLATES = [
    "[커뮤니티] {k1} 논란…{dom} 표현 확산",
    "댓글창 뒤흔든 {k1}…{dom}성 문구 도마",
    "{k1} 둘러싼 설전…{dom} 표현에 비판 커져",
    "확산되는 {dom} 표현, {k1} 관련 발언이 불씨",
    "{k1} 언급 이후 파장…{dom}성 표현 논란",
    "온라인서 번진 {k1}…{dom}성 발언에 우려",
    "‘{k1}’ 키워드로 갈등 격화…{dom} 표현 문제",
    "{k1} 이슈에 엇갈린 반응…{dom} 표현 재확산",
    "{k1} 관련 게시물 확산…{dom}성 문구에 신고 잇따라",
    "논쟁 키운 {k1}…{dom} 표현이 운영 리스크로",
    # 필요 시 더 추가하세요(50~100개면 거의 중복이 사라집니다)
]

# 간단 불용어(필요하면 확장)
STOPWORDS = set([
    "은","는","이","가","을","를","에","에서","에게","와","과","도","만","로","으로","하고","한","하다",
    "그","저","것","수","등","및","또","더","너무","진짜","정말","왜","뭐","좀","다","내","네"
])

def stable_seed(text: str) -> int:
    return int(hashlib.md5(text.encode("utf-8")).hexdigest()[:8], 16)

def tokenize(text: str):
    # 형태소 분석 없이도 중복 해소에 충분한 수준의 토큰화
    text = re.sub(r"[^\w가-힣\s]", " ", text)
    tokens = [t for t in text.split() if len(t) >= 2 and t not in STOPWORDS]
    return tokens

def pick_keywords(text: str, n=3):
    toks = tokenize(text)
    if not toks:
        return ["논란", "발언", "확산"][:n]
    # 빈도 기반(단순) + 재현 가능한 셔플
    # 너무 긴 텍스트가 아니라면 이 정도로도 충분히 다양해집니다.
    uniq = list(dict.fromkeys(toks))  # 순서 유지 중복 제거
    return uniq[:n] if len(uniq) >= n else (uniq + ["논란","확산","발언"])[:n]

def make_title(text: str, dominant: str, harm_score: int) -> str:
    rng = random.Random(stable_seed(text))
    kws = pick_keywords(text, n=3)
    k1 = kws[0]
    tpl = rng.choice(TITLE_TEMPLATES)

    # 강도 슬롯(선택): 과다 유해를 제목에도 은근히 반영 가능
    if harm_score >= 6:
        intensity = "과열"
    elif harm_score >= 4:
        intensity = "확산"
    else:
        intensity = "논란"

    title = tpl.format(k1=k1, dom=dominant)
    # 약간의 변주를 추가(재현 가능)
    suffix_pool = ["", f"…{intensity} 조짐", "…운영 대응 필요", "…신고 증가"]
    title = title + rng.choice(suffix_pool)
    return title.strip()

def assign_category(text: str) -> str:
    h = int(hashlib.md5(text.encode("utf-8")).hexdigest(), 16)
    return CATEGORIES[h % len(CATEGORIES)]

# =========================
# LOAD
# =========================
df = pd.read_csv(STAGE2_PATH)
df["문장"] = df["문장"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df["문장"].str.len() >= MIN_LEN].copy()

harm_cols = [c for c in df.columns if c not in ["문장", "정상"]]
df["harm_score"] = df[harm_cols].sum(axis=1)

# =========================
# IMPACT4 CANDIDATES
# =========================
# 기본: 복합 유해(>=4)
cand = df[(df["정상"] == 0) & (df["harm_score"] >= 4)].copy()
cand = cand.drop_duplicates(subset=["문장"]).copy()

# (선택) 고위험 라벨 포함 조건을 추가하고 싶으면 아래를 사용하세요.
# 실제 컬럼명이 존재하는지 먼저 확인 필요.
# existing_high = [c for c in HIGH_RISK_LABELS_CANDIDATES if c in cand.columns]
# if existing_high:
#     cand = cand[cand[existing_high].sum(axis=1) >= 1].copy()

if len(cand) < TARGET_N:
    raise RuntimeError(f"후보 부족: {len(cand)} < {TARGET_N} (임계값 완화 필요)")

cand["dominant"] = cand[harm_cols].idxmax(axis=1)

cand["newsCategory"] = cand["문장"].apply(assign_category)
cand["newsSubcategory"] = cand["newsCategory"]
cand["newsContent"] = cand["문장"]
cand["useType"] = 0

# 제목 생성
cand["newsTitle"] = cand.apply(lambda r: make_title(r["문장"], r["dominant"], int(r["harm_score"])), axis=1)

# =========================
# SAMPLING (단순 랜덤 + 카테고리 균형)
# =========================
# 7개 카테고리 균등에 가깝게
per_cat = TARGET_N // len(CATEGORIES)
remainder = TARGET_N % len(CATEGORIES)

picked = []
for i, cat in enumerate(CATEGORIES):
    n = per_cat + (1 if i < remainder else 0)
    sub = cand[cand["newsCategory"] == cat]
    if len(sub) < n:
        picked.append(sub)
    else:
        picked.append(sub.sample(n=n, random_state=42))

sel = pd.concat(picked, ignore_index=True)

# 부족분은 전체에서 보충
if len(sel) < TARGET_N:
    need = TARGET_N - len(sel)
    rest = cand[~cand["문장"].isin(sel["문장"])]  # 이미 뽑힌 문장 제외 (ignore_index 후에는 index로 구분 불가)
    sel = pd.concat([sel, rest.sample(n=need, random_state=42)], ignore_index=True)

# =========================
# TITLE DEDUP SAFETY: 제목 중복이 남으면 템플릿을 바꿔 재생성
# =========================
# (템플릿 풀을 충분히 키우면 거의 안 걸립니다)
seen = set()
new_titles = []
for _, r in sel.iterrows():
    t = r["newsTitle"]
    if t not in seen:
        seen.add(t)
        new_titles.append(t)
        continue
    # 중복이면 템플릿을 강제로 바꿔가며 재생성
    rng = random.Random(stable_seed(r["문장"]) + 99991)
    kws = pick_keywords(r["문장"], n=3)
    k1 = kws[0]
    for _try in range(50):
        tpl = rng.choice(TITLE_TEMPLATES)
        suffix_pool = ["", "…후폭풍", "…논쟁 재점화", "…운영 리스크"]
        t2 = tpl.format(k1=k1, dom=r["dominant"]) + rng.choice(suffix_pool)
        t2 = t2.strip()
        if t2 not in seen:
            t = t2
            break
    seen.add(t)
    new_titles.append(t)

sel["newsTitle"] = new_titles

out = sel[["newsCategory","newsSubcategory","newsTitle","newsContent","useType"]].drop_duplicates(subset=["newsContent"]).head(TARGET_N)

os.makedirs(OUT_DIR, exist_ok=True)
csv_path = os.path.join(OUT_DIR, "impact4_dataset_500.csv")
json_path = os.path.join(OUT_DIR, "impact4_dataset_500.json")

out.to_csv(csv_path, index=False, encoding="utf-8-sig")
out.to_json(json_path, orient="records", force_ascii=False, indent=2)

print(f"[OK] rows={len(out)}")
print("[CHECK] title unique:", out["newsTitle"].nunique(), "/", len(out))
print("[CHECK] category dist:\n", out["newsCategory"].value_counts())
print("[EXAMPLE TITLES]\n", "\n".join(out["newsTitle"].head(10).tolist()))
